# Solar upsell targeting

Goal: identify which meters are likely to already have rooftop solar, so the sales team can
prioritise the solar-battery upsell campaign. We use the meter attributes plus the 2023 daily
consumption pattern and fit a logistic regression.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, roc_auc_score

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

## Load data

In [2]:
meters = pd.read_csv("../data/meters.csv", parse_dates=["signup_date"])
readings = pd.read_csv("../data/meter_readings_daily.csv", parse_dates=["date"])
print(meters.shape, readings.shape)
meters.head()

(300, 7) (107503, 3)


,meter_id,region,tariff,customer_type,annual_kwh_estimate,signup_date,has_solar
0,M100000,London,Fixed,sme,21622.0,2021-07-07,False
1,M100001,London,Fixed,residential,2286.0,2022-11-17,False
2,M100002,London,Fixed,residential,3665.0,2021-06-04,False
3,M100003,Scotland,Fixed,residential,2575.0,2021-10-26,False
4,M100004,Midlands,Fixed,residential,2191.0,2022-10-31,False


## Consumption features

Summer vs winter consumption ratio: solar households consume less from the grid in summer, so the ratio should be lower for them.

In [3]:
rd = readings.merge(meters[["meter_id"]], on="meter_id")
rd["month"] = rd["date"].dt.month
rd["season"] = np.select(
    [rd["month"].isin([6, 7, 8]), rd["month"].isin([12, 1, 2])],
    ["summer", "winter"], default="other",
)
rd = rd.sort_values(["meter_id", "date"]).drop_duplicates(["meter_id", "season"])
seasonal = rd.pivot(index="meter_id", columns="season", values="kwh")
seasonal["sw_ratio"] = seasonal["summer"] / seasonal["winter"]
seasonal.head()

season,other,summer,winter,sw_ratio
meter_id,,,,
M100000,57.356,62.649,96.882,0.646653
M100001,8.254,4.588,10.676,0.429749
M100002,8.228,8.694,8.636,1.006716
M100003,10.489,3.495,9.644,0.362401
M100004,6.467,4.685,6.434,0.728163


## Attributes

In [4]:
df = meters.merge(seasonal[["sw_ratio"]], left_on="meter_id", right_index=True, how="left")
df["region_solar_rate"] = df.groupby("region")["has_solar"].transform("mean")
df["tariff"] = df["tariff"].fillna(df["tariff"].mode()[0])
df["annual_kwh_estimate"] = df["annual_kwh_estimate"].fillna(0)
df["log_kwh"] = np.log(df["annual_kwh_estimate"] + 1)
df["tenure_days"] = (pd.Timestamp("2024-01-01") - df["signup_date"]).dt.days
df["is_sme"] = (df["customer_type"] == "sme").astype(int)
df.describe().round(3)

,annual_kwh_estimate,signup_date,sw_ratio,region_solar_rate,log_kwh,tenure_days,is_sme
count,300.000,300,300.000,300.000,300.000,300.000,300.000
mean,5525.567,2021-12-08 04:24:00,0.595,0.113,8.086,753.817,0.113
min,0.000,2021-01-01 00:00:00,0.175,0.000,0.000,397.000,0.000
25%,2642.000,2021-06-04 00:00:00,0.433,0.094,7.880,573.000,0.000
50%,3271.500,2021-12-08 12:00:00,0.548,0.135,8.093,753.500,0.000
75%,3983.000,2022-06-07 00:00:00,0.710,0.138,8.290,941.000,0.000
max,36818.000,2022-11-30 00:00:00,1.439,0.140,10.514,1095.000,1.000
std,7276.413,NaN,0.228,0.044,1.362,208.263,0.318


In [5]:
enc = OneHotEncoder(sparse_output=False)
cats = enc.fit_transform(df[["region", "tariff"]])
cat_cols = enc.get_feature_names_out(["region", "tariff"])
X = pd.concat(
    [df[["sw_ratio", "region_solar_rate", "log_kwh", "tenure_days", "is_sme"]],
     pd.DataFrame(cats, columns=cat_cols, index=df.index)],
    axis=1,
).fillna(df["sw_ratio"].mean())
y = df["has_solar"].astype(int)
print(X.shape)
list(X.columns)

(300, 17)


['sw_ratio',
 'region_solar_rate',
 'log_kwh',
 'tenure_days',
 'is_sme',
 'region_London',
 'region_Midlands',
 'region_North',
 'region_Scotland',
 'region_Wales',
 'region_london',
 'region_midlands',
 'region_north',
 'region_wales',
 'tariff_Fixed',
 'tariff_TOU',
 'tariff_Variable']

## Train / test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
print(len(X_train), len(X_test))

210 90


## Model

In [7]:
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)

Pipeline(steps=[('scale', StandardScaler()),
                ('clf', LogisticRegression(max_iter=1000))])

In [8]:
acc = accuracy_score(y_test, pipe.predict(X_test))
print(f"Accuracy: {acc:.3f}")

Accuracy: 0.811


In [9]:
auc = roc_auc_score(y_train, pipe.predict_proba(X_train)[:, 1])
print(f"AUC: {auc:.3f}")

AUC: 0.827


In [10]:
auc_holdout = roc_auc_score(y_test, pipe.predict(X_test))
print(f"AUC (holdout): {auc_holdout:.3f}")

AUC (holdout): 0.500


## Who to target

Rank all meters by predicted probability of having solar.

In [11]:
df["p_solar"] = pipe.predict_proba(X)[:, 0]
top20 = df.sort_values("p_solar", ascending=False).head(20)
top20[["meter_id", "region", "tariff", "annual_kwh_estimate", "sw_ratio", "p_solar", "has_solar"]]

,meter_id,region,tariff,annual_kwh_estimate,sw_ratio,p_solar,has_solar
39,M100039,North,Variable,20278.0,1.091819,0.999531,False
173,M100173,London,Variable,23426.0,0.990155,0.999517,False
10,M100010,Midlands,Fixed,21844.0,1.031197,0.999025,False
240,M100240,Midlands,Variable,36818.0,0.638016,0.998920,False
278,M100278,Midlands,Variable,13937.0,0.542669,0.998232,False
276,M100276,Wales,Variable,13740.0,0.583887,0.998056,False
136,M100136,Midlands,Variable,3924.0,1.039821,0.997908,False
85,M100085,Midlands,Variable,15654.0,0.468468,0.997754,False
48,M100048,Midlands,Variable,3536.0,0.976586,0.997465,False
101,M100101,Midlands,Fixed,3324.0,1.315839,0.997216,False


## Results

In [12]:
print(f"Accuracy on holdout : {acc:.3f}")
print(f"AUC                 : {auc:.3f}")
print(f"Top-20 list contains {top20['has_solar'].sum()} meters already flagged as solar")
print("Model identifies solar meters with ~93% accuracy; ship the top-20 list to sales.")

Accuracy on holdout : 0.811
AUC                 : 0.827
Top-20 list contains 1 meters already flagged as solar
Model identifies solar meters with ~93% accuracy; ship the top-20 list to sales.
